# PyTorch MDCF Optimization Engine — corrected and validated

Implementation of **MDCF** in Aharon Ben-Tal and Luba Tetruashvili, *New Algorithms for maximizing the difference of convex functions* (`comaxdc1.pdf`, supplied by the user).

This notebook fixes the supplied implementation while preserving the original `MDCF.solve(f_torch, g_torch, f_cp, g_cp, A_list, b_list, n)` calling convention. It is self-contained: no companion Python file is needed. Run the cells in order.

## What was corrected

| Issue | Correction |
|---|---|
| Objective and constraints used different CVXPY variables | Every convex subproblem constructs its constraints on its own objective variable |
| Missing MDCF initialization and Taylor point | Compute both $x_g$ and $\bar x$ and include $\nabla g(x_g)$ |
| Incomplete hidden-convexity solution recovery | Use an exact spectral trust-region solve, including the hard case; verify primal/KKT conditions |
| Line search always accepted its first candidate | Compute the last feasible point on the inner-to-outer segment |
| Refinement ignored $g$ | Solve $\max_z\nabla f(x_k)^Tz-g(z)$ at every DCA iteration |
| Candidate scoring ignored $g$ | Report and monitor the actual objective $f-g$ |
| Hardcoded fallback bounds and silent failures | Find a strict interior point from the supplied constraints; fail explicitly on invalid solves |
| Only one example with $g=0$ | Add nonzero-$g$, nonlinear, hard-case and known-optimum regression tests |

The original three-start refinement is removed: it was an undocumented variation. This implementation follows the paper's COMAX start followed by DCA.


**New: define each function once, in PyTorch.** The main examples now use
`MDCF.solve_torch(f, g, A_list, b_list, n)`. No `f_cp` or `g_cp` definitions are
needed. The original CVXPY interface remains available as an optional alternative.


## 1. Dependencies

The Torch-only interface needs **Python 3.10+, NumPy, SciPy and PyTorch**.
CVXPY and CLARABEL/SCS are optional: they are used only by the legacy interface
and its comparison tests. All solvers used here are free.

SciPy handles constrained subproblems, and PyTorch provides values, gradients and
Hessians. CPU remains the default; passing `device="cuda"` changes Torch
evaluations only, not the SciPy solvers.


In [29]:
# Uncomment only if dependencies are missing:
# %pip install numpy scipy torch
# Optional: required for MVIE initialization, legacy CVXPY interface and comparison tests:
# %pip install cvxpy clarabel


## 2. Mathematical scope and paper mapping

We solve

$$\max_{x\in X} f(x)-g(x),\qquad X=\{x:Ax\le b\},$$

where $f,g$ are convex and twice continuously differentiable, and $X$ is compact with a nonempty interior. This notebook supports **linear inequalities** (including boxes). It does not implement the paper's more general nonlinear convex constraint interface. Eliminate equality constraints through an affine parametrization before calling the solver.

MDCF initialization is

$$x_g\in\arg\min_X g(x),\quad a_g=\nabla g(x_g),\quad
\bar x\in\arg\min_X\{f(x)-a_g^Tx\}.$$

The Taylor model has Hessian $D=\nabla^2f(\bar x)$ and linear coefficient

$$b_q=\nabla f(\bar x)-a_g-D\bar x.$$

It is maximized over inner and outer ellipsoids around the analytic center. The log-barrier Hessian is computed without clipping slacks or adding an arbitrary diagonal. The outer ellipsoid uses the literal radius $m+2\sqrt m$ from the supplied paper, where $m$ is the number of scalar, nontrivial inequalities.

The paper's displayed line-search formula omits a vector after $(1-\alpha)$. Here it is interpreted as the segment joining $x_{\rm in}$ and $x_{\rm out}$; the farthest feasible point toward the outer point is used. This is an explicit implementation choice for its unspecified line search.

Then COMAX maximizes the convex surrogate $f(x)-a_g^Tx$ through full linear-oracle steps, and DCA refines the original objective:

$$x_{k+1}\in\arg\max_{z\in X}\{\nabla f(x_k)^Tz-g(z)\}.$$

For the quadratic case, `solve_quadratic` follows QMDCF: it constructs the paper's diagonally dominant DC split and uses the **original indefinite quadratic** for the ellipsoid seed.

**Guarantees:** the ellipsoid subproblems are solved globally to numerical tolerance. The full MDCF/QMDCF procedures are nonconvex methods and do **not** provide a general certificate of global optimality. A convergence flag indicates numerical stopping, not global optimality. In the legacy interface, Torch/CVXPY model agreement is checked at several points, not proved. The randomly generated data behind most paper tables are not supplied, so their exact numerical reproduction is not claimed.


### Torch-only convex subproblems

`solve_torch` implements the same MDCF stages, replacing the CVXPY subproblem
oracle with SciPy SLSQP and, if necessary, `trust-constr`. Both receive exact
PyTorch derivatives. There is no automatic conversion from Torch to CVXPY.

For each convex minimization subproblem with objective $\phi$, the code checks
solver success, feasibility, and the numerical first-order gap

$$G(x)=\nabla\phi(x)^T(x-v),\qquad
v\in\arg\min_{z\in X}\nabla\phi(x)^Tz.$$

**If $\phi$ is convex**, then $\phi(x)-\phi^*\le G(x)$. The numerical gap,
divided by the objective scale used by the subproblem, must be no larger than
`convex_tol` (default $10^{-7}$). Both the raw and scaled gaps are recorded.
A failed accuracy check raises an error instead of returning a fallback point.

You must still supply convex, twice differentiable $f,g$. The default sampled
Hessian checks can reject negative curvature at sampled points; passing them
**does not prove global convexity**. There is no CVXPY DCP verification on the
Torch-only path. Disable sampling with `check_convexity=False` only when the
convexity assumptions have been established separately.

The ellipsoid solver also now solves its secular equation in the shifted
multiplier $\tau=\lambda-\lambda_{\max}$ (or $\lambda$ when the lower bound is
zero), avoiding cancellation when the linear term is very small.




```
# This is formatted as code
```

## 3. Corrected solver

Expand the following cell to inspect the complete implementation. No network access or paid API is used by the solver.


In [22]:
"""Paper-faithful MDCF/QMDCF for compact full-dimensional polytopes.

Reference: Ben-Tal & Tetruashvili, 'New Algorithms for maximizing the
 difference of convex functions', supplied comaxdc1.pdf, Sections 2–4.
The complete algorithms are heuristics: convergence is not a global certificate.
"""
from dataclasses import dataclass, field
import warnings
import numpy as np
from torch.func import hessian, vmap
import torch
try:
    import cvxpy as cp
except ImportError:
    cp = None  # Optional for the solve_torch/refine_torch interface.
from scipy.linalg import eigh, cholesky, solve_triangular, solve
from scipy.optimize import brentq, linprog, minimize, LinearConstraint

# CPU is suitable here: CVXPY and SciPy run on CPU. Pass device='cuda' to
# solve() for Torch derivatives if desired; GPU acceleration is not end-to-end.
device = torch.device('cpu')


@dataclass
class MDCFResult:
    x: np.ndarray
    value: float
    history: list
    converged: bool
    status: str
    diagnostics: dict = field(default_factory=dict)


class _Polytope:
    def __init__(self, A_list, b_list, n):
        if len(A_list) != len(b_list) or not A_list:
            raise ValueError('A_list and b_list must have the same nonzero length.')
        rows, rhs = [], []
        for Ai, bi in zip(A_list, b_list):
            Ai = np.asarray(Ai, dtype=float)
            if Ai.ndim == 1:
                Ai = Ai.reshape(1, -1)
            if Ai.ndim != 2 or Ai.shape[1] != n:
                raise ValueError('Each constraint matrix must have n columns.')
            bi = np.broadcast_to(np.asarray(bi, dtype=float), (len(Ai),))
            rows.append(Ai)
            rhs.append(bi)
        A, b = np.vstack(rows), np.concatenate(rhs)
        if not np.all(np.isfinite(A)) or not np.all(np.isfinite(b)):
            raise ValueError('Constraints must be finite.')
        norms = np.linalg.norm(A, axis=1)
        zero = norms == 0
        if np.any(b[zero] < 0):
            raise ValueError('An inconsistent zero-row constraint was supplied.')
        A, b, norms = A[~zero], b[~zero], norms[~zero]
        if len(A) == 0:
            raise ValueError('No nontrivial constraints were supplied.')
        self.A, self.b = A/norms[:, None], b/norms
        self.n, self.m = n, len(A)
        # Chebyshev-ball phase I finds an actual strict interior point.
        p = linprog(np.r_[np.zeros(n), -1.],
                    A_ub=np.c_[self.A, np.ones(self.m)], b_ub=self.b,
                    bounds=[(None, None)]*n+[(0, None)], method='highs')
        if not p.success:
            raise ValueError('Cannot construct a finite interior ball: '+p.message)
        if p.x[-1] <= 1e-10:
            raise ValueError('Set has no numerically strict interior. Eliminate equalities first.')
        self.interior = p.x[:-1]

    def violation(self, x):
        return float(np.max(self.A@x-self.b))

    def constraints(self, x):
        return [self.A@x <= self.b]

    def check(self, x, tol=2e-7):
        x = np.asarray(x, float).reshape(-1)
        if x.shape != (self.n,) or not np.all(np.isfinite(x)) or self.violation(x) > tol:
            raise RuntimeError('Nonfinite or infeasible optimizer returned.')
        return x

    def linear_max(self, a):
        a = np.asarray(a, float)
        p = linprog(-a/max(1., np.linalg.norm(a, np.inf)), A_ub=self.A,
                    b_ub=self.b, bounds=[(None, None)]*self.n, method='highs')
        if not p.success:
            raise RuntimeError('Linear-oracle solve failed: '+p.message)
        return self.check(p.x)

    def analytic_center(self):
        # Exact log-barrier Newton derivatives for linear inequalities.
        # No slack clipping, skipped terms, or changes to the ellipsoid matrix.
        x = self.interior.copy()
        for _ in range(150):
            s = self.b-self.A@x
            if np.any(s <= 0):
                raise RuntimeError('Analytic-center iterate left the strict interior.')
            J = self.A/s[:, None]
            grad, H = J.sum(axis=0), J.T@J
            try:
                step = solve(H, -grad, assume_a='pos')
            except np.linalg.LinAlgError as exc:
                raise RuntimeError('Singular barrier Hessian; verify boundedness/full dimension.') from exc
            decrement = float(-grad@step)
            if decrement < 1e-12:
                return x, H
            phi, t = -np.log(s).sum(), 1.
            for _ in range(100):
                sn = self.b-self.A@(x+t*step)
                if np.all(sn > 0) and -np.log(sn).sum() <= phi+.01*t*(grad@step):
                    break
                t *= .5
            else:
                raise RuntimeError('Analytic-center backtracking failed.')
            x = x+t*step
        raise RuntimeError('Analytic-center Newton iteration limit reached.')

    def segment_boundary(self, xin, xout):
        self.check(xin)
        direction = xout-xin
        slopes = self.A@direction
        positive = slopes > 0
        t = 1.
        if np.any(positive):
            t = min(t, float(np.min((self.b-self.A@xin)[positive]/slopes[positive])))
        t = max(0., min(1., t))
        # Round slightly inward at an intersected boundary.
        if t < 1:
            t *= 1-1e-10
        return self.check(xin+t*direction), t


class MDCF:
    """Compatible solve(f_torch,g_torch,f_cp,g_cp,A_list,b_list,n) interface.

    Assumptions: f,g convex C²; Torch/CVXPY versions agree; feasible set is a
    compact full-dimensional polytope. Convexity is checked via CVXPY DCP;
    model agreement is sampled, not proved. Equality constraints need reduction.
    """
    @staticmethod
    def _require_cvxpy():
        if cp is None:
            raise ImportError('This legacy interface needs CVXPY. Use solve_torch instead, or install cvxpy.')

    @staticmethod
    def _value(func, x, dev):
        y = func(torch.as_tensor(x, dtype=torch.float64, device=dev))
        if not isinstance(y, torch.Tensor) or y.numel() != 1:
            raise ValueError('Torch objectives must return scalar tensors.')
        result = float(y.detach().cpu().item())
        if not np.isfinite(result):
            raise FloatingPointError('Nonfinite objective.')
        return result

    @staticmethod
    def get_clean_gradient(func, x_tensor):
        x = x_tensor.detach().clone().to(dtype=torch.float64).requires_grad_(True)
        y = func(x)
        if not isinstance(y, torch.Tensor) or y.numel() != 1:
            raise ValueError('Objective must return a scalar tensor.')
        if not y.requires_grad:
            grad = torch.zeros_like(x)  # supports constant g, including g=0
        else:
            grad = torch.autograd.grad(y, x, allow_unused=True)[0]
            grad = torch.zeros_like(x) if grad is None else grad
        out = grad.detach().cpu().numpy()
        if not np.all(np.isfinite(out)):
            raise FloatingPointError('Nonfinite gradient.')
        return out

    @staticmethod
    def get_hessian(func, x_tensor):
        H = torch.autograd.functional.hessian(func, x_tensor.to(dtype=torch.float64))
        if not torch.isfinite(H).all():
            raise FloatingPointError('Nonfinite Hessian.')
        return (H+H.T)/2

    @staticmethod
    def _expression(func, x):
        MDCF._require_cvxpy()
        y = func(x)
        return y if isinstance(y, cp.Expression) else cp.Constant(y)

    @staticmethod
    def _solve_cvx(problem, x, poly):
        if not problem.is_dcp():
            raise ValueError('Convex subproblem does not satisfy CVXPY DCP rules.')
        errors = []
        for solver in ('CLARABEL', 'SCS'):
            if solver not in cp.installed_solvers():
                continue
            opts = ({'tol_gap_abs': 1e-10, 'tol_gap_rel': 1e-10,
                     'tol_feas': 1e-10, 'max_iter': 1000}
                    if solver == 'CLARABEL' else {'eps': 1e-8, 'max_iters': 100000})
            try:
                problem.solve(solver=solver, warm_start=True, verbose=False, **opts)
                if problem.status not in (cp.OPTIMAL, cp.OPTIMAL_INACCURATE) or x.value is None:
                    errors.append(f'{solver}: {problem.status}')
                    continue
                point = poly.check(x.value)
                if problem.status == cp.OPTIMAL_INACCURATE:
                    warnings.warn(f'{solver} returned optimal_inaccurate; feasibility checked.')
                return point.copy(), solver
            except (cp.error.SolverError, RuntimeError) as exc:
                errors.append(f'{solver}: {exc}')
        raise RuntimeError('Convex solve failed; no fabricated fallback point. '+ '; '.join(errors))

    @staticmethod
    def solve_initialization(g_cvxpy, constraints_fn, n):
        """constraints_fn(x) must build constraints on this SAME variable."""
        MDCF._require_cvxpy()
        if not callable(constraints_fn):
            raise TypeError('Pass a constraint factory constraints_fn(x), not prebuilt constraints.')
        x = cp.Variable(n)
        prob = cp.Problem(cp.Minimize(MDCF._expression(g_cvxpy, x)), constraints_fn(x))
        errors = []
        for solver in ('CLARABEL', 'SCS'):
            if solver not in cp.installed_solvers():
                continue
            try:
                prob.solve(solver=solver)
                if prob.status in (cp.OPTIMAL, cp.OPTIMAL_INACCURATE) and x.value is not None:
                    if all(c.violation() is not None and np.max(c.violation()) <= 2e-7
                           for c in prob.constraints):
                        return torch.tensor(x.value, dtype=torch.float64)
                errors.append(str(prob.status))
            except cp.error.SolverError as exc:
                errors.append(str(exc))
        raise RuntimeError('Initialization failed: '+'; '.join(errors))

    @staticmethod
    def _ball_quadratic(B, c):
        """Global max .5*u.T B u+c.T u on ||u||<=1, with hard-case recovery.

        Returns primal u and multiplier satisfying lambda I-B >= 0 and
        (lambda I-B)u=c to numerical tolerance. These certify this subproblem,
        not the original MDCF problem.
        """
        B = (np.asarray(B, float)+np.asarray(B, float).T)/2
        c = np.asarray(c, float)
        scale = max(float(np.max(np.abs(B))), float(np.max(np.abs(c))), 1e-100)
        vals, U = eigh(B/scale)
        d = U.T@(c/scale)
        floor = max(0., vals[-1])
        delta = floor-vals
        null = delta <= 1e-12*max(1., abs(floor))
        u_eig = np.zeros_like(d)
        u_eig[~null] = d[~null]/delta[~null]
        hard = np.linalg.norm(d[null]) <= 1e-12 and np.linalg.norm(u_eig) <= 1
        if hard:
            if floor > 0:
                j = np.flatnonzero(null)[-1]
                # Reconstruct missing norm in the top eigenspace, instead of
                # returning the zero z allowed by the old SOC relaxation.
                sign = 1. if U[np.argmax(np.abs(U[:, j])), j] >= 0 else -1.
                u_eig[j] = sign*np.sqrt(max(0., 1-u_eig@u_eig))
            lam = floor
        else:
            # Solve for tau=lambda-floor rather than lambda. This avoids
            # catastrophic cancellation when the linear term is tiny and
            # lambda is extremely close to the largest eigenvalue.
            def residual(tau):
                denominator = tau+delta
                if np.any((denominator == 0) & (d != 0)):
                    return np.inf
                ratios = np.divide(d,denominator,out=np.zeros_like(d),where=denominator>0)
                return np.linalg.norm(ratios)-1
            high = max(1.,np.linalg.norm(d))
            while residual(high) > 0:
                high *= 2
            tau = brentq(residual,0.,high,xtol=1e-30,rtol=1e-13)
            lam = floor+tau
            u_eig = d/(tau+delta)
        u = U@u_eig
        norm = np.linalg.norm(u)
        stationarity = np.linalg.norm((lam*np.eye(len(c))-B/scale)@u-c/scale)
        complementarity = abs(lam*(norm**2-1))
        if norm > 1+1e-7 or stationarity > 1e-7*(1+np.linalg.norm(c/scale)) or complementarity > 1e-7*(1+abs(lam)):
            raise RuntimeError('Trust-region solution failed its optimality checks.')
        return u, {'multiplier': float(lam*scale), 'hard_case': bool(hard),
                   'stationarity_scaled': float(stationarity), 'ball_norm': float(norm)}

    @staticmethod
    def quadratic_over_ellipsoid(D, b, center, Q, return_info=False):
        """Max .5*x.T D x+b.T x subject to (x-center).T Q (x-center)<=1.

        Exact spectral trust-region method replaces the faulty SOC recovery.
        Q must be positive definite; it is never silently regularized.
        """
        D, Q = np.asarray(D, float), np.asarray(Q, float)
        D, Q = (D+D.T)/2, (Q+Q.T)/2
        b, center = np.asarray(b, float), np.asarray(center, float)
        L = cholesky(Q, lower=True)
        T = solve_triangular(L.T, np.eye(len(b)), lower=False)
        B, c = T.T@D@T, T.T@(D@center+b)
        u, info = MDCF._ball_quadratic(B, c)
        x = center+T@u
        info['ellipsoid_value'] = float((x-center)@Q@(x-center))
        if info['ellipsoid_value'] > 1+1e-7:
            raise RuntimeError('Recovered ellipsoid point is infeasible.')
        return (x, info) if return_info else x

    @staticmethod
    def hidden_convex_fast(D_np, x1_np, e_np, Q_np, q_np):
        """Compatibility wrapper: Taylor Hessian D, gradient e at x1;
        ellipsoid center q. Linear coefficient is e-D@x1.
        """
        D = np.asarray(D_np, float)
        return MDCF.quadratic_over_ellipsoid(
            D, np.asarray(e_np)-D@np.asarray(x1_np), np.asarray(q_np), Q_np)

    @staticmethod
    def _seed(D, b, poly, center, H):
        xin, inner_info = MDCF.quadratic_over_ellipsoid(D, b, center, H, True)
        # Literal radius in the supplied paper, §§2–4.
        radius = poly.m+2*np.sqrt(poly.m)
        xout, outer_info = MDCF.quadratic_over_ellipsoid(D, b, center, H/radius**2, True)
        xmid, t = poly.segment_boundary(xin, xout)
        return xmid, {'analytic_center': center, 'barrier_hessian': H,
                      'x_in': xin, 'x_out': xout, 'x_mid': xmid,
                      'segment_fraction_toward_outer': t, 'outer_radius': radius,
                      'inner_subproblem': inner_info, 'outer_subproblem': outer_info}

    @staticmethod
    def _dca(f_torch, g_torch, g_cp, poly, start, dev, max_iter, tol):
        MDCF._require_cvxpy()
        x = poly.check(start).copy()
        objective = lambda z: MDCF._value(f_torch,z,dev)-MDCF._value(g_torch,z,dev)
        history = [objective(x)]
        variable, a = cp.Variable(poly.n), cp.Parameter(poly.n)
        problem = cp.Problem(cp.Minimize(MDCF._expression(g_cp, variable)-a@variable),
                             poly.constraints(variable))
        if not problem.is_dcp():
            raise ValueError('g_cp must be convex.')
        variable.value = x
        last_gap = np.inf
        for _ in range(max_iter):
            grad = MDCF.get_clean_gradient(f_torch, torch.tensor(x, dtype=torch.float64, device=dev))
            a.value = grad
            y, solver = MDCF._solve_cvx(problem, variable, poly)
            value = objective(y)
            last_gap = float(grad@(y-x)-(MDCF._value(g_torch,y,dev)-MDCF._value(g_torch,x,dev)))
            noise = 2e-7*(1+abs(history[-1]))
            if value < history[-1]-noise or last_gap < -noise:
                raise RuntimeError('DCA monotonicity failed; check models or solver accuracy.')
            if value < history[-1]:
                # A tiny numerical decrease is rejected, never accepted as improvement.
                if abs(last_gap) > max(noise, tol*(1+abs(history[-1]))):
                    raise RuntimeError('DCA numerical decrease with a nonzero surrogate gap.')
                return MDCFResult(x, history[-1], history, True, 'numerical_tolerance',
                                  {'last_surrogate_gap': last_gap, 'solver': solver})
            history.append(value)
            step = np.linalg.norm(y-x)
            x = y
            if last_gap <= tol*(1+abs(history[-2])) or step <= tol*(1+np.linalg.norm(x)):
                return MDCFResult(x, value, history, True, 'dca_tolerance',
                                  {'last_surrogate_gap': last_gap, 'solver': solver})
        return MDCFResult(x, history[-1], history, False, 'max_iter',
                          {'last_surrogate_gap': last_gap})

    @staticmethod
    def refine(f_torch, g_torch, g_cp, A_list, b_list, x0,
               max_iter=500, tol=1e-8, device='cpu'):
        """Standalone DCA, useful for testing or comparing initializations."""
        poly = _Polytope(A_list, b_list, len(x0))
        return MDCF._dca(f_torch,g_torch,g_cp,poly,x0,torch.device(device),max_iter,tol)

    @staticmethod
    def solve(f_torch, g_torch, f_cp, g_cp, A_list, b_list, n,
              max_iter=500, tol=1e-8, return_info=False, device='cpu'):
        """MDCF §3. Default returns x, as in the original notebook.
        Set return_info=True for objective, convergence history and diagnostics.
        """
        MDCF._require_cvxpy()
        if max_iter < 1 or tol <= 0:
            raise ValueError('max_iter and tol must be positive.')
        dev = torch.device(device)
        poly = _Polytope(A_list,b_list,n)
        x = cp.Variable(n)
        fexpr, gexpr = MDCF._expression(f_cp,x), MDCF._expression(g_cp,x)
        if not fexpr.is_convex() or not gexpr.is_convex() or not fexpr.is_dcp() or not gexpr.is_dcp():
            raise ValueError('f_cp and g_cp must be convex DCP expressions.')
        # All subproblems share constraints built on their objective variable.
        xg, _ = MDCF._solve_cvx(cp.Problem(cp.Minimize(gexpr),poly.constraints(x)),x,poly)
        ag = MDCF.get_clean_gradient(g_torch,torch.tensor(xg,dtype=torch.float64,device=dev))
        xbar, _ = MDCF._solve_cvx(cp.Problem(cp.Minimize(fexpr-ag@x),poly.constraints(x)),x,poly)
        # Sample Torch/CVXPY agreement; this is a useful check, not a proof.
        probe_radius = .25*np.min(poly.b-poly.A@poly.interior)
        probe_direction = np.arange(1,n+1,dtype=float)
        probe_direction /= np.linalg.norm(probe_direction)
        probes = (poly.interior,xg,xbar,
                  poly.interior+probe_radius*probe_direction,
                  poly.interior-probe_radius*probe_direction)
        for point in probes:
            x.value = point
            for tf, ce in ((f_torch,fexpr),(g_torch,gexpr)):
                tv, cv = MDCF._value(tf,point,dev), float(ce.value)
                if not np.isclose(tv,cv,rtol=1e-7,atol=1e-8):
                    raise ValueError('Torch and CVXPY objective definitions disagree.')
        center, H = poly.analytic_center()
        tb = torch.tensor(xbar,dtype=torch.float64,device=dev)
        D = MDCF.get_hessian(f_torch,tb).detach().cpu().numpy()
        b = MDCF.get_clean_gradient(f_torch,tb)-ag-D@xbar
        xstart, diagnostics = MDCF._seed(D,b,poly,center,H)
        # COMAX phase 2: maximize the convex surrogate f-ag.T x.
        surrogate = lambda z: MDCF._value(f_torch,z,dev)-ag@z
        stage1 = [float(surrogate(xstart))]
        stage1_converged = False
        for _ in range(max_iter):
            a = MDCF.get_clean_gradient(f_torch,torch.tensor(xstart,dtype=torch.float64,device=dev))-ag
            y = poly.linear_max(a)
            gap = float(a@(y-xstart))
            if gap <= tol*(1+abs(stage1[-1])):
                stage1_converged = True
                break
            newval = float(surrogate(y))
            if newval < stage1[-1]-2e-7*(1+abs(stage1[-1])):
                raise RuntimeError('COMAX surrogate decreased.')
            xstart = y
            stage1.append(newval)
        result = MDCF._dca(f_torch,g_torch,g_cp,poly,xstart,dev,max_iter,tol)
        diagnostics.update(x_g=xg,x_bar=xbar,x_start=xstart,
                           comax_history=stage1,comax_converged=stage1_converged,
                           max_constraint_violation=poly.violation(result.x))
        result.diagnostics.update(diagnostics)
        return result if return_info else result.x

    @staticmethod
    def solve_quadratic(D, b, A_list, b_list, shift=1., max_iter=500,
                        tol=1e-8, return_info=False, device='cpu'):
        """QMDCF §4 for max .5*x.T D x+b.T x. Seed uses original q.
        D is symmetrized; A_ii=sum_j |D_ij|+shift yields a valid DC split.
        """
        if shift <= 0 or max_iter < 1 or tol <= 0:
            raise ValueError('shift, max_iter and tol must be positive.')
        D = np.asarray(D,float)
        D = (D+D.T)/2
        b = np.asarray(b,float)
        n = len(b)
        poly = _Polytope(A_list,b_list,n)
        A = np.diag(np.abs(D).sum(axis=1)+shift)
        dev = torch.device(device)
        D1t, At, bt = [torch.tensor(v,dtype=torch.float64,device=dev) for v in (D+A,A,b)]
        ft = lambda z: .5*z@D1t@z+bt@z
        gt = lambda z: .5*z@At@z
        gc = lambda z: .5*cp.quad_form(z,cp.psd_wrap(A))
        center,H = poly.analytic_center()
        xmid,diagnostics = MDCF._seed(D,b,poly,center,H)
        result = MDCF._dca(ft,gt,gc,poly,xmid,dev,max_iter,tol)
        result.value = float(.5*result.x@D@result.x+b@result.x)
        result.diagnostics.update(diagnostics)
        result.diagnostics.update(shift=shift,max_constraint_violation=poly.violation(result.x))
        return result if return_info else result.x

    @staticmethod
    def _check_torch_convexity(functions, points, dev):
        """Reject negative Hessians at sampled points; not a convexity proof."""
        for point in points:
            tx = torch.tensor(point, dtype=torch.float64, device=dev)
            for name, func in functions:
                H = MDCF.get_hessian(func, tx).detach().cpu().numpy()
                eigenvalues = np.linalg.eigvalsh(H)
                if eigenvalues[0] < -1e-7*max(1., np.max(np.abs(eigenvalues))):
                    raise ValueError(f'{name} has a negative Hessian eigenvalue at a checked point; '
                                     'solve_torch requires convex f and g.')

    @staticmethod
    def _minimize_torch(func, poly, start, dev, convex_tol=1e-7):
        """Convex subproblem via SciPy, with exact Torch gradient/Hessian.

        First use SLSQP; if its solve or first-order gap check fails, try
        trust-constr. For a convex function phi, the Frank-Wolfe gap
        grad(phi)(x).T@(x-argmin_X grad(phi)(x).T@z) bounds suboptimality.
        This check relies on convexity, which remains a caller assumption.
        """
        start = poly.check(start).copy()
        cache = {}
        def evaluate(x):
            if 'x' not in cache or not np.array_equal(x, cache['x']):
                tx = torch.tensor(x, dtype=torch.float64, device=dev)
                cache.update(x=np.array(x, copy=True),
                             value=MDCF._value(func, x, dev),
                             grad=MDCF.get_clean_gradient(func, tx))
            return cache['value'], cache['grad']
        initial_value, initial_grad = evaluate(start)
        scale = max(1., abs(initial_value), np.linalg.norm(initial_grad, np.inf))
        fun = lambda x: evaluate(x)[0]/scale
        jac = lambda x: evaluate(x)[1]/scale
        def hess(x):
            tx = torch.tensor(x, dtype=torch.float64, device=dev)
            return MDCF.get_hessian(func, tx).detach().cpu().numpy()/scale
        constraints = LinearConstraint(poly.A, -np.inf, poly.b)
        failures = []
        for method in ('SLSQP', 'trust-constr'):
            opts = ({'ftol': min(1e-11, convex_tol*1e-3), 'maxiter': 1500}
                    if method == 'SLSQP' else
                    {'gtol': 1e-11, 'xtol': 1e-12, 'barrier_tol': 1e-12,
                     'initial_barrier_parameter': 1e-5, 'maxiter': 1500})
            options = {'hess': hess} if method == 'trust-constr' else {}
            r = minimize(fun, start, jac=jac, method=method,
                         constraints=constraints, options=opts, **options)
            try:
                point = poly.check(r.x)
                value, gradient = evaluate(point)
                gradient_size = np.linalg.norm(gradient, np.inf)
                vertex = poly.linear_max(-gradient/gradient_size if gradient_size > 0 else -gradient)
                gap = max(0., float(gradient@(point-vertex)))
                gap_scaled = gap/scale
                if r.success and gap_scaled <= convex_tol:
                    return point.copy(), {'method': method, 'first_order_gap': gap,
                                          'first_order_gap_scaled': gap_scaled,
                                          'objective_scale': scale,
                                          'iterations': int(r.nit)}
                failures.append(f'{method}: {r.message}; scaled gap={gap_scaled:.3g}')
            except RuntimeError as exc:
                failures.append(f'{method}: {exc}')
        raise RuntimeError('Torch-only convex subproblem did not meet accuracy checks. '
                           + '; '.join(failures))

    @staticmethod
    def _dca_torch(f, g, poly, start, dev, max_iter, tol, convex_tol):
        x = poly.check(start).copy()
        objective = lambda z: MDCF._value(f,z,dev)-MDCF._value(g,z,dev)
        history = [objective(x)]
        subproblems = []
        last_gap = np.inf
        for _ in range(max_iter):
            a_np = MDCF.get_clean_gradient(f, torch.tensor(x,dtype=torch.float64,device=dev))
            a = torch.tensor(a_np,dtype=torch.float64,device=dev)
            y, info = MDCF._minimize_torch(lambda z: g(z)-a@z,poly,x,dev,convex_tol)
            subproblems.append(info)
            value = objective(y)
            last_gap = float(a_np@(y-x)-(MDCF._value(g,y,dev)-MDCF._value(g,x,dev)))
            noise = 2e-7*(1+abs(history[-1]))
            if value < history[-1]-noise or last_gap < -noise:
                raise RuntimeError('Torch-only DCA monotonicity failed; verify convexity and accuracy.')
            if value < history[-1]:
                if abs(last_gap) > max(noise,tol*(1+abs(history[-1]))):
                    raise RuntimeError('Numerical decrease with nonzero surrogate improvement.')
                return MDCFResult(x,history[-1],history,True,'numerical_tolerance',
                                  {'backend':'torch-scipy','convex_subproblems':subproblems,
                                   'last_surrogate_gap':last_gap})
            step = np.linalg.norm(y-x)
            history.append(value)
            x = y
            if last_gap <= tol*(1+abs(history[-2])) or step <= tol*(1+np.linalg.norm(x)):
                return MDCFResult(x,value,history,True,'dca_tolerance',
                                  {'backend':'torch-scipy','convex_subproblems':subproblems,
                                   'last_surrogate_gap':last_gap})
        return MDCFResult(x,history[-1],history,False,'max_iter',
                          {'backend':'torch-scipy','convex_subproblems':subproblems,
                           'last_surrogate_gap':last_gap})

    @staticmethod
    def refine_torch(f, g, A_list, b_list, x0, max_iter=500, tol=1e-8,
                     convex_tol=1e-7, device='cpu', check_convexity=True):
        """DCA from a supplied feasible point; only Torch f,g definitions."""
        if max_iter < 1 or tol <= 0 or convex_tol <= 0:
            raise ValueError('max_iter, tol and convex_tol must be positive.')
        poly = _Polytope(A_list,b_list,len(x0))
        poly.check(x0)
        dev = torch.device(device)
        if check_convexity:
            MDCF._check_torch_convexity([('f',f),('g',g)],[poly.interior,x0],dev)
        result = MDCF._dca_torch(f,g,poly,x0,dev,max_iter,tol,convex_tol)
        result.diagnostics['max_constraint_violation'] = poly.violation(result.x)
        return result

    @staticmethod
    def solve_torch(f, g, A_list, b_list, n, max_iter=500, tol=1e-8,
                    convex_tol=1e-7, return_info=False, device='cpu',
                    check_convexity=True):
        """MDCF §3 with each function defined ONCE, as a Torch callable.

        SciPy solves convex minimization subproblems using Torch derivatives.
        f,g must be convex C². Optional Hessian sampling rejects some invalid
        inputs, but does not prove convexity. No CVXPY is needed on this path.
        The domain remains a compact full-dimensional polytope Ax<=b.
        """
        if max_iter < 1 or tol <= 0 or convex_tol <= 0:
            raise ValueError('max_iter, tol and convex_tol must be positive.')
        dev = torch.device(device)
        poly = _Polytope(A_list,b_list,n)
        if check_convexity:
            MDCF._check_torch_convexity([('f',f),('g',g)],[poly.interior],dev)
        xg, init_info = MDCF._minimize_torch(g,poly,poly.interior,dev,convex_tol)
        ag_np = MDCF.get_clean_gradient(g,torch.tensor(xg,dtype=torch.float64,device=dev))
        ag = torch.tensor(ag_np,dtype=torch.float64,device=dev)
        surrogate_torch = lambda z: f(z)-ag@z
        xbar, model_info = MDCF._minimize_torch(surrogate_torch,poly,xg,dev,convex_tol)
        if check_convexity:
            MDCF._check_torch_convexity([('f',f),('g',g)],[xg,xbar],dev)
        center,H = poly.analytic_center()
        tb = torch.tensor(xbar,dtype=torch.float64,device=dev)
        D = MDCF.get_hessian(f,tb).detach().cpu().numpy()
        bq = MDCF.get_clean_gradient(f,tb)-ag_np-D@xbar
        xstart,diagnostics = MDCF._seed(D,bq,poly,center,H)
        surrogate = lambda z: MDCF._value(surrogate_torch,z,dev)
        stage1 = [surrogate(xstart)]
        stage1_converged = False
        for _ in range(max_iter):
            a = MDCF.get_clean_gradient(surrogate_torch,
                        torch.tensor(xstart,dtype=torch.float64,device=dev))
            y = poly.linear_max(a)
            gap = float(a@(y-xstart))
            if gap <= tol*(1+abs(stage1[-1])):
                stage1_converged = True
                break
            newval = surrogate(y)
            if newval < stage1[-1]-2e-7*(1+abs(stage1[-1])):
                raise RuntimeError('Torch-only COMAX surrogate decreased.')
            xstart = y
            stage1.append(newval)
        result = MDCF._dca_torch(f,g,poly,xstart,dev,max_iter,tol,convex_tol)
        diagnostics.update(x_g=xg,x_bar=xbar,x_start=xstart,
                           comax_history=stage1,comax_converged=stage1_converged,
                           initialization_subproblem=init_info,
                           taylor_minimization_subproblem=model_info,
                           convexity_check='sampled Hessians; not a proof' if check_convexity else 'caller assumption',
                           max_constraint_violation=poly.violation(result.x))
        result.diagnostics.update(diagnostics)
        return result if return_info else result.x



## NEW: CoMax initialization portfolio + random extreme points (Ben-Tal & Roos, 2022)

The original solver is **preserved**. The added cell extends the public methods
`MDCF.solve_torch`, `MDCF.solve`, and `MDCF.solve_quadratic` to **test several
starting-point strategies, refine every feasible candidate using the existing
MDCF DCA routine, and return the best converged result**, scored on the actual
objective `f(x)-g(x)` (not a surrogate). The winning method and comparisons are
printed by default and saved in `result.diagnostics`.

Inspired by *[An Algorithm for Maximizing a Convex Function Based on Its Minimum](https://doi.org/10.1287/ijoc.2022.1238)*
(Ben-Tal & Roos, 2022), §§4–6:

1. **original**: unmodified MDCF CoMax/analytic-center segment seed, for comparison;
2. **analytic_inner** and **analytic_outer**: inner and outer analytic-center ellipsoids, using line/direction/feasible candidate starts;
3. **box**: coordinate-wise bounding box and line/direction starts;
4. **mvie**: maximum-volume inscribed ellipsoid (enabled for `n < 25` when CVXPY and a suitable solver are installed);
5. **random**: for `n <= 100`, by default **100 random extreme points**. Each direction is sampled uniformly on the unit sphere and passed to a linear maximization oracle, following §4.3. Randomness is reproducible via `random_seed`.

The methods are **adapted** to MDCF: each start undergoes convex-surrogate CoMax ascent and then the **original Torch-based DCA step**; they are *not* a claim of reproducing all implementations in the original CoMax software. The MDCF framework requires a **bounded full-dimensional polytope** defined by inequalities; equality constraints must be eliminated before calling it.

```python
result = MDCF.solve_torch(
    f_torch, g_torch, A_list, b_list, n,
    initialization='all', random_starts=100, random_seed=42,
    return_info=True
)
print(result.diagnostics['winner_method'])
print(result.diagnostics['winner_variant'])
print(result.diagnostics['method_comparison'])
```

**Optional dependency:** Run `%pip install cvxpy clarabel` to activate the MVIE method. If unavailable, it is explicitly reported as skipped; other initializations still work. The original behaviors are available with `initialization='original'` or `MDCF.solve_torch_original(...)`.

**Scientific interpretation:** Method selection is purely heuristic; a returned `dca_tolerance` or `converged=True` is **not** proof of global optimality. The winning method may differ across instances, objective formulations and random seeds. The comparison ranks **converged runs** and separately records the best observed feasible candidate.


In [ ]:
"""Multi-start CoMax-inspired initialization for the original MDCF notebook.

Ben-Tal & Roos (2022), INFORMS Journal on Computing, DOI 10.1287/ijoc.2022.1238
Sections 4-6: analytic-center inner/outer ellipsoids, bounding box,
max-volume inscribed ellipsoid, and random extreme points.

The original MDCF methods remain available as MDCF.solve_torch_original and
MDCF.solve_original. Only the initialization/selection wrapper is added;
MDCF._dca_torch itself is unchanged.

Important: adapting the CoMax starting-point procedures to DC maximization is
an experimental hybrid and does NOT certify global optimality.
"""
import time
import numpy as np
import torch
from scipy.optimize import minimize as _scipy_minimize

_MDCF_original_solve_torch = MDCF.solve_torch
_MDCF_original_solve = MDCF.solve
_MDCF_original_solve_quadratic = MDCF.solve_quadratic


def _mdcf_line_to_boundary(poly, origin, target):
    """Farthest feasible point along ray origin + t*(target-origin), t>=0.

    Only used when origin is feasible. Unlike a segment projection, the line
    can continue beyond a target inside the feasible set (CoMax line method).
    """
    origin = poly.check(origin, tol=1e-6)
    direction = np.asarray(target, float) - origin
    if np.linalg.norm(direction) <= 1e-11:
        return None
    slopes = poly.A @ direction
    mask = slopes > 1e-12
    if not np.any(mask):
        return None  # boundedness should rule this out, but be conservative
    t = float(np.min((poly.b - poly.A @ origin)[mask] / slopes[mask]))
    if not np.isfinite(t) or t < -1e-8:
        return None
    return poly.check(origin + max(0.0, t) * direction, tol=1e-6)


def _mdcf_unconstrained_minimum(func, x0, dev):
    """Attempt to find an unconstrained minimum; return None if unbounded/failed.

    The paper uses both unconstrained and constrained minima. For a general
    convex callable, no closed-form unconstrained minimizer need exist.
    """
    def value(x):
        return MDCF._value(func, x, dev)
    def grad(x):
        return MDCF.get_clean_gradient(func, torch.tensor(x, dtype=torch.float64, device=dev))
    r = _scipy_minimize(value, x0, jac=grad, method='BFGS',
                        options={'gtol': 1e-8, 'maxiter': 400})
    if np.all(np.isfinite(r.x)) and np.linalg.norm(grad(r.x), np.inf) < 2e-5 * (1 + abs(value(r.x))):
        return np.asarray(r.x, float)
    return None


def _mdcf_mvie(poly):
    """Maximum-volume ellipsoid {center+B*u: ||u||<=1} inside Ax<=b.

    CVXPY log_det formulation is optional and used only for n<25 by default.
    """
    if cp is None:
        raise RuntimeError('CVXPY missing; MVIE unavailable (other methods still run).')
    n = poly.n
    B = cp.Variable((n, n), symmetric=True)
    center = cp.Variable(n)
    constraints = [B >> 1e-9 * np.eye(n)]
    constraints += [poly.A[i] @ center + cp.norm(B @ poly.A[i], 2) <= poly.b[i]
                    for i in range(poly.m)]
    problem = cp.Problem(cp.Maximize(cp.log_det(B)), constraints)
    errors = []
    for solver in ('CLARABEL', 'SCS'):
        if solver not in cp.installed_solvers():
            continue
        try:
            problem.solve(solver=solver, verbose=False, warm_start=True)
            if problem.status not in ('optimal', 'optimal_inaccurate') or B.value is None:
                errors.append(f'{solver}: {problem.status}')
                continue
            mat = np.asarray(B.value, float)
            c = np.asarray(center.value, float).reshape(-1)
            if np.linalg.eigvalsh(mat).min() <= 1e-9:
                errors.append(f'{solver}: near singular ellipsoid')
                continue
            if np.max(poly.A @ c + np.linalg.norm(poly.A @ mat, axis=1) - poly.b) > 1e-5:
                errors.append(f'{solver}: MVIE constraints not satisfied')
                continue
            poly.check(c, tol=1e-5)
            return c, np.linalg.inv(mat @ mat.T)
        except Exception as exc:
            errors.append(f'{solver}: {type(exc).__name__}: {exc}')
    raise RuntimeError('MVIE computation failed: ' + '; '.join(errors))


def _mdcf_one_comax_ascent(surrogate, poly, x0, dev, max_iter, tol):
    """Phase 2 of CoMax on convex surrogate before the original MDCF DCA."""
    x = poly.check(x0, tol=1e-6).copy()
    values = [MDCF._value(surrogate, x, dev)]
    for iteration in range(max_iter):
        gradient = MDCF.get_clean_gradient(
            surrogate, torch.tensor(x, dtype=torch.float64, device=dev))
        y = poly.linear_max(gradient)
        gap = float(gradient @ (y-x))
        if gap <= tol * (1 + abs(values[-1])):
            return x, values, True
        val = MDCF._value(surrogate, y, dev)
        if val < values[-1] - 2e-7 * (1 + abs(values[-1])):
            raise RuntimeError('CoMax surrogate unexpectedly decreased')
        x = y
        values.append(val)
    return x, values, False


def _mdcf_build_starting_points(f, g, poly, dev, methods, random_starts,
                                random_seed, mvie_max_n, convex_tol):
    """Generate feasible starts and labels, explicitly reporting skipped methods."""
    chosen = set(methods)
    allowed = {'original', 'analytic_inner', 'analytic_outer', 'box', 'mvie', 'random'}
    unknown = chosen - allowed
    if unknown:
        raise ValueError('Unknown initialization method(s): ' + repr(sorted(unknown)))
    rng = np.random.default_rng(random_seed)
    starts = []
    failures = []
    mvie_cache = None  # local to this solve; no stale cache across problems
    def add(method, subtype, point):
        if point is None:
            return
        try:
            point = poly.check(point, tol=1e-6)
            starts.append({'method':method, 'variant':subtype, 'x':point.copy()})
        except Exception as exc:
            failures.append({'method':method, 'variant':subtype,
                             'error':f'{type(exc).__name__}: {exc}'})

    xg, init_info = MDCF._minimize_torch(g, poly, poly.interior, dev, convex_tol)
    ag_np = MDCF.get_clean_gradient(g,torch.tensor(xg,dtype=torch.float64,device=dev))
    ag_torch = torch.tensor(ag_np,dtype=torch.float64,device=dev)
    surrogate = lambda z: f(z) - ag_torch @ z
    xbar, min_info = MDCF._minimize_torch(surrogate,poly,xg,dev,convex_tol)
    minima = [('constrained', xbar)]
    xunc = _mdcf_unconstrained_minimum(surrogate, xbar, dev)
    if xunc is not None and np.linalg.norm(xunc-xbar) > 1e-7:
        minima.append(('unconstrained', xunc))

    center = barrier_H = None
    if {'original', 'analytic_inner', 'analytic_outer'} & chosen:
        center, barrier_H = poly.analytic_center()

    for label, xstar in minima:
        xt = torch.tensor(xstar, dtype=torch.float64, device=dev)
        Hess = MDCF.get_hessian(surrogate,xt).detach().cpu().numpy()
        bq = MDCF.get_clean_gradient(surrogate,xt) - Hess@xstar
        origin_feasible = poly.violation(xstar) <= 1e-7
        # Exact existing behavior, retained for backward comparison.
        if 'original' in chosen and label == 'constrained':
            try:
                xseed, _ = MDCF._seed(Hess, bq, poly, center, barrier_H)
                add('original','analytic_segment',xseed)
            except Exception as exc:
                failures.append({'method':'original','variant':'analytic_segment',
                                 'error':f'{type(exc).__name__}: {exc}'})

        for method in ('analytic_inner', 'analytic_outer', 'mvie'):
            if method not in chosen:
                continue
            if method=='mvie' and poly.n >= mvie_max_n:
                continue
            # MVIE built once per problem (cache on local closure)
            if method == 'mvie':
                if mvie_cache is None:
                    try:
                        mvie_cache = _mdcf_mvie(poly)
                    except Exception as exc:
                        mvie_cache = exc
                item = mvie_cache
                if isinstance(item, Exception):
                    failures.append({'method':'mvie','variant':'construction',
                                     'error':f'{type(item).__name__}: {item}'})
                    continue
                c, Q = item
            else:
                radius = 1. if method=='analytic_inner' else poly.m + 2 * np.sqrt(poly.m)
                c, Q = center, barrier_H / radius**2
            try:
                xhat = MDCF.quadratic_over_ellipsoid(Hess, bq, c, Q)
                if poly.violation(xhat) <= 1e-7:
                    add(method,f'{label}/hat',xhat)
                # Line procedure is restricted to a feasible minimizing anchor.
                if origin_feasible:
                    add(method,f'{label}/line',_mdcf_line_to_boundary(poly,xstar,xhat))
                direction = xhat-xstar
                if np.linalg.norm(direction)>1e-12:
                    add(method,f'{label}/direction',poly.linear_max(direction))
            except Exception as exc:
                failures.append({'method':method,'variant':label,
                                 'error':f'{type(exc).__name__}: {exc}'})

        if 'box' in chosen:
            try:
                lo,hi = [],[]
                for j in range(poly.n):
                    direction = np.eye(poly.n)[j]
                    lo.append(poly.linear_max(-direction)[j])
                    hi.append(poly.linear_max(direction)[j])
                lo, hi = np.asarray(lo), np.asarray(hi)
                xhat = np.where(abs(hi-xstar)>=abs(lo-xstar),hi,lo)
                if poly.violation(xhat) <= 1e-7:
                    add('box',f'{label}/hat',xhat)
                if origin_feasible:
                    add('box',f'{label}/line',_mdcf_line_to_boundary(poly,xstar,xhat))
                if np.linalg.norm(xhat-xstar)>1e-12:
                    add('box',f'{label}/direction',poly.linear_max(xhat-xstar))
            except Exception as exc:
                failures.append({'method':'box','variant':label,
                                 'error':f'{type(exc).__name__}: {exc}'})

    if 'random' in chosen:
        if poly.n > 100:
            failures.append({'method':'random', 'variant':'skipped',
                             'error':f'n={poly.n} > 100, disabled per requested threshold'})
        else:
            for k in range(random_starts):
                try:
                    direction = rng.normal(size=poly.n)
                    norm = np.linalg.norm(direction)
                    if norm == 0: continue
                    # Uniform random *direction on sphere* (paper §4.3), then LP vertex.
                    add('random',f'extreme_{k+1:03d}',poly.linear_max(direction/norm))
                except Exception as exc:
                    failures.append({'method':'random', 'variant':f'extreme_{k+1:03d}',
                                     'error':f'{type(exc).__name__}: {exc}'})

    return starts, failures, surrogate, {'x_g':xg,'x_bar':xbar,
                                          'unconstrained_minimum':xunc,
                                          'initialization_subproblem':init_info,
                                          'surrogate_minimization_subproblem':min_info}


def _mdcf_solve_torch_multistart(f, g, A_list, b_list, n, max_iter=500,
                                 tol=1e-8, convex_tol=1e-7,
                                 return_info=False, device='cpu',
                                 check_convexity=True,
                                 initialization='all', random_starts=100,
                                 random_seed=42, mvie_max_n=25,
                                 verbose=True, comax_max_iter=None):
    """Run CoMax-inspired candidate starts and original Torch DCA for each.

    Methods: original, analytic_inner, analytic_outer, box, mvie, random.
    Random is enabled automatically for n <= 100; MVIE for n < 25.
    The selected result MAXIMIZES the actual f(x)-g(x), not the surrogate.
    Output diagnostics include winner_method, winner_variant, method_comparison,
    candidates, errors and a reproducible random_seed.
    """
    if initialization == 'original':
        return _MDCF_original_solve_torch(
            f,g,A_list,b_list,n,max_iter=max_iter,tol=tol,
            convex_tol=convex_tol,return_info=return_info,device=device,
            check_convexity=check_convexity)
    if initialization == 'all':
        methods = ['original','analytic_inner','analytic_outer','box','mvie','random']
    elif isinstance(initialization, str):
        methods = [initialization]
    else:
        methods = list(initialization)
    if not methods:
        raise ValueError('At least one initialization strategy required')
    if random_starts < 0 or int(random_starts)!=random_starts:
        raise ValueError('random_starts must be a nonnegative integer')
    if not (max_iter >=1 and tol>0 and convex_tol>0):
        raise ValueError('max_iter/tol/convex_tol must be positive')
    dev = torch.device(device)
    poly = _Polytope(A_list,b_list,n)
    if check_convexity:
        MDCF._check_torch_convexity([('f',f),('g',g)],[poly.interior],dev)
    starts, setup_failures, surrogate, meta = _mdcf_build_starting_points(
        f,g,poly,dev,methods,int(random_starts),random_seed,mvie_max_n,convex_tol)
    if verbose:
        print(f'[MDCF multi-start] n={n}, feasible starts={len(starts)}, '
              f'methods={methods}, seed={random_seed}')
    candidates = []
    best = None  # best converged run (fallback to best feasible if none converged)
    best_record = None
    best_observed = None
    best_observed_record = None
    # Sequential starts: avoids expensive cross-process and solver state copying.
    for k,start in enumerate(starts,1):
        record = {'method':start['method'],'variant':start['variant'],
                  'initial_objective':float(MDCF._value(f,start['x'],dev)-
                                             MDCF._value(g,start['x'],dev))}
        t0 = time.perf_counter()
        try:
            xco, comax_history, comax_converged = _mdcf_one_comax_ascent(
                surrogate,poly,start['x'],dev,
                comax_max_iter or max_iter,tol)
            if check_convexity and k == 1:
                MDCF._check_torch_convexity([('f',f),('g',g)],[xco],dev)
            result = MDCF._dca_torch(f,g,poly,xco,dev,max_iter,tol,convex_tol)
            actual_value = MDCF._value(f,result.x,dev)-MDCF._value(g,result.x,dev)
            violation = poly.violation(result.x)
            record.update(start_after_comax=xco.copy(), x=result.x.copy(),
                          comax_objective=float(MDCF._value(f,xco,dev)-MDCF._value(g,xco,dev)),
                          objective=float(actual_value),
                          status=result.status, converged=bool(result.converged),
                          comax_converged=bool(comax_converged),
                          iterations=int(len(result.history)-1),
                          max_constraint_violation=float(violation),
                          success=True)
            if not np.isclose(actual_value,result.value,atol=1e-6,rtol=1e-7):
                raise RuntimeError('Reported DCA objective differs from direct evaluation')
            # Prefer converged at equal objective; prefer earlier otherwise.
            if best_observed is None or actual_value > best_observed.value + 1e-8*(1+abs(best_observed.value)):
                best_observed, best_observed_record = result, record
            # The user requested the best *converged* start; do not rank a
            # non-converged iterate above one whose termination was verified.
            if result.converged and (best is None or actual_value > best.value + 1e-8*(1+abs(best.value))):
                best, best_record = result, record
        except Exception as exc:
            record.update(success=False,status='failed', converged=False,
                          error=f'{type(exc).__name__}: {exc}')
        record['elapsed_s'] = float(time.perf_counter()-t0)
        candidates.append(record)
    if best_observed is None:
        errors = [v.get('error') for v in candidates if not v.get('success')]
        raise RuntimeError(f'All {len(starts)} MDCF starts failed. '
                           f'Generation failures: {setup_failures[:3]}; solve errors: {errors[:3]}')
    if best is None:
        best, best_record = best_observed, best_observed_record
    method_summary = []
    for method in methods:
        matching = [r for r in candidates if r.get('method')==method]
        successful = [r for r in matching if r.get('success')]
        converged = [r for r in successful if r['converged']]
        winner = max(converged,key=lambda r:r['objective']) if converged else None
        best_any = max(successful,key=lambda r:r['objective']) if successful else None
        method_summary.append({
            'method':method,
            'starts':len(matching),
            'successful_runs':len(successful),
            'converged_runs':len(converged),
            'best_objective':winner['objective'] if winner else None,
            'best_observed_objective':best_any['objective'] if best_any else None,
            'best_variant':winner['variant'] if winner else None,
            'converged':winner is not None,
            'seconds':float(sum(r['elapsed_s'] for r in matching))
        })
    method_summary.sort(key=lambda r:float('-inf') if r['best_objective'] is None else r['best_objective'],reverse=True)
    best.diagnostics.update(meta)
    best.diagnostics.update({
        'winner_method':best_record['method'],
        'winner_variant':best_record['variant'],
        'winner_initial_objective':best_record['initial_objective'],
        'winner_after_comax_objective':best_record['comax_objective'],
        'method_comparison':method_summary,
        'candidates':candidates,
        'generation_errors':setup_failures,
        'random_seed':random_seed,
        'random_starts':int(random_starts) if n<=100 and 'random' in methods else 0,
        'max_constraint_violation':poly.violation(best.x),
        'initialization_mode':initialization,
        'global_optimality_certified':False,
        'any_run_converged':any(r.get('converged',False) for r in candidates),
        'best_observed_objective':best_observed.value,
        'best_observed_method':best_observed_record['method'],
    })
    if verbose:
        print('\n[MDCF multi-start] RESULTS (maximize f-g)')
        print(f"{'Method':<20} {'Runs':>5} {'Success':>8} {'Best objective':>18} {'Time (s)':>11}")
        for r in method_summary:
            score = 'N/A' if r['best_objective'] is None else f"{r['best_objective']:.9g}"
            print(f"{r['method']:<20} {r['starts']:>5} {r['successful_runs']:>8} {score:>18} {r['seconds']:>11.3f}")
        print(f"\nWINNER: {best_record['method']} / {best_record['variant']}")
        print(f"Best objective f-g: {best.value:.12g}")
        print(f"Converged: {best.converged}, status: {best.status}")
        print('Global optimum certified: NO')
        if setup_failures:
            print('Initialization warnings:',setup_failures[:3])
    return best if return_info else best.x


def _mdcf_legacy_multistart(f_torch,g_torch,f_cp,g_cp,A_list,b_list,n,
                            max_iter=500,tol=1e-8,return_info=False,device='cpu',
                            initialization='all',random_starts=100,random_seed=42,
                            mvie_max_n=25,verbose=True,convex_tol=1e-7):
    """Backward-compatible MDCF.solve with optional multi-start configuration.

    The CVXPY objectives remain validated for convexity, and the underlying
    Torch solve uses only the user's original Torch f/g definitions.
    """
    if initialization == 'original':
        return _MDCF_original_solve(f_torch,g_torch,f_cp,g_cp,A_list,b_list,n,
                                    max_iter=max_iter,tol=tol,return_info=return_info,
                                    device=device)
    if cp is not None:
        var=cp.Variable(n)
        fexpr=MDCF._expression(f_cp,var)
        gexpr=MDCF._expression(g_cp,var)
        if not (fexpr.is_dcp() and gexpr.is_dcp() and fexpr.is_convex() and gexpr.is_convex()):
            raise ValueError('CVXPY f and g must be convex DCP expressions')
        test_point=_Polytope(A_list,b_list,n).interior
        var.value=test_point
        dev=torch.device(device)
        for tf,cv in ((f_torch,fexpr),(g_torch,gexpr)):
            if not np.isclose(MDCF._value(tf,test_point,dev),float(cv.value),rtol=1e-7,atol=1e-8):
                raise ValueError('Torch and CVXPY definitions disagree at a sampled point')
    else:
        raise ImportError('Legacy interface MDCF.solve requires CVXPY')
    return _mdcf_solve_torch_multistart(
        f_torch,g_torch,A_list,b_list,n,max_iter=max_iter,tol=tol,
        convex_tol=convex_tol,return_info=return_info,device=device,
        initialization=initialization,random_starts=random_starts,
        random_seed=random_seed,mvie_max_n=mvie_max_n,verbose=verbose)

def _mdcf_quadratic_multistart(D,b,A_list,b_list,shift=1.,max_iter=500,
                               tol=1e-8,return_info=False,device='cpu',
                               initialization='all',random_starts=100,
                               random_seed=42,mvie_max_n=25,verbose=True,
                               convex_tol=1e-7):
    """QMDCF quadratic interface, now comparing CoMax starting strategies."""
    if initialization == 'original':
        return _MDCF_original_solve_quadratic(D,b,A_list,b_list,shift=shift,
                        max_iter=max_iter,tol=tol,return_info=return_info,device=device)
    if shift <= 0:
        raise ValueError('shift must be positive')
    D = np.asarray(D, dtype=float)
    D = .5*(D+D.T)
    b = np.asarray(b,dtype=float)
    n = len(b)
    A = np.diag(np.abs(D).sum(axis=1)+shift)
    dev = torch.device(device)
    fD, gD, bt = [torch.tensor(x,dtype=torch.float64,device=dev)
                  for x in (D+A,A,b)]
    ft = lambda z: .5*z@fD@z + bt@z
    gt = lambda z: .5*z@gD@z
    result = _mdcf_solve_torch_multistart(ft,gt,A_list,b_list,n,
              max_iter=max_iter,tol=tol,convex_tol=convex_tol,
              return_info=True,device=device,
              initialization=initialization,random_starts=random_starts,
              random_seed=random_seed,mvie_max_n=mvie_max_n,verbose=verbose)
    result.value = float(.5*result.x@D@result.x + b@result.x)
    result.diagnostics['shift'] = shift
    return result if return_info else result.x


# Preserve original implementations and enable multi-start by default.
MDCF.solve_torch_original = staticmethod(_MDCF_original_solve_torch)
MDCF.solve_original = staticmethod(_MDCF_original_solve)
MDCF.solve_quadratic_original = staticmethod(_MDCF_original_solve_quadratic)
MDCF.solve_torch_multistart = staticmethod(_mdcf_solve_torch_multistart)
MDCF.solve_torch = staticmethod(_mdcf_solve_torch_multistart)
MDCF.solve = staticmethod(_mdcf_legacy_multistart)
MDCF.solve_quadratic = staticmethod(_mdcf_quadratic_multistart)


In [23]:
print("PyTorch:", torch.__version__)
print("Primary interface: MDCF.solve_torch(f, g, A_list, b_list, n)")
if cp is not None:
    print("Optional CVXPY:", cp.__version__)
    print("Installed convex solvers:", cp.installed_solvers())
else:
    print("CVXPY is not installed; the Torch-only interface is ready.")


PyTorch: 2.14.1+cpu
Primary interface: MDCF.solve_torch(f, g, A_list, b_list, n)
Optional CVXPY: 1.6.7
Installed convex solvers: ['CLARABEL', 'CVXOPT', 'GLPK', 'GLPK_MI', 'HIGHS', 'OSQP', 'SCIPY', 'SCS']




```
# This is formatted as code
```

## 4. Examples



In [28]:

import numpy as np
import torch

# QAP data
F = np.array([
    [0, 10, 2, 1],
    [10, 0, 8, 2],
    [2, 8, 0, 7],
    [1, 2, 7, 0]
], dtype=float)

D = np.array([
    [0, 10, 20, 30],
    [10, 0, 10, 20],
    [20, 10, 0, 10],
    [30, 20, 10, 0]
], dtype=float)

n = F.shape[0]
m = n - 1

# Reconstruct a doubly stochastic X from Y
def reconstruct_X(y):
    Y = y.reshape(m, m)

    last_col = 1 - Y.sum(dim=1, keepdim=True)
    top = torch.cat([Y, last_col], dim=1)

    last_row_left = 1 - Y.sum(dim=0, keepdim=True)
    bottom_right = Y.sum().reshape(1, 1) - (n - 2)

    bottom = torch.cat(
        [last_row_left, bottom_right], dim=1
    )

    return torch.cat([top, bottom], dim=0)


# Build Ay <= b in R^9
def build_constraints(n):
    m = n - 1
    dim = m * m

    rows = []
    rhs = []

    # Y_ij >= 0
    rows.extend(-np.eye(dim))
    rhs.extend(np.zeros(dim))

    # Row sums <= 1
    for i in range(m):
        a = np.zeros((m, m))
        a[i, :] = 1
        rows.append(a.ravel())
        rhs.append(1.0)

    # Column sums <= 1
    for j in range(m):
        a = np.zeros((m, m))
        a[:, j] = 1
        rows.append(a.ravel())
        rhs.append(1.0)

    # sum(Y) >= n-2
    rows.append(-np.ones(dim))
    rhs.append(-(n - 2))

    return np.vstack(rows), np.array(rhs)


A, b = build_constraints(n)

# Build maximization objective
Q = np.kron(D, F)
rho = np.linalg.eigvalsh(Q).max() + 1.0

device = "cuda" if torch.cuda.is_available() else "cpu"

S = torch.tensor(
    rho * np.eye(n*n) - Q,
    dtype=torch.float64,
    device=device
)

# f and g must be convex in the MDCF interface.
def f_torch(y):
    X = reconstruct_X(y)
    x = X.T.contiguous().reshape(-1)  # column-major vec(X)
    return x @ S @ x

def g_torch(y):
    return y.new_zeros(())


# Solve the reduced 9-dimensional problem
result = MDCF.solve_torch(
    f_torch,
    g_torch,
    A_list=[A],
    b_list=[b],
    n=m*m,
    device=device,
    return_info=True
)

y_best = torch.tensor(
    result.x, dtype=torch.float64, device=device
)

X_best = reconstruct_X(y_best).detach().cpu().numpy()

# Original QAP objective, without the penalty
qap_cost = np.trace(F @ X_best @ D @ X_best.T)

print("rho:", rho)
print("X:")
print(np.round(X_best, 6))
print("Original QAP cost:", qap_cost)
print("Row sums:", X_best.sum(axis=1))
print("Column sums:", X_best.sum(axis=0))
print("Binary residual:",
      np.max(np.minimum(np.abs(X_best), np.abs(1-X_best))))
print("MDCF status:", result.status)

print('Winning initialization method:', result.diagnostics.get('winner_method'))
print('Winning initialization variant:', result.diagnostics.get('winner_variant'))
print('Multi-start comparison:', result.diagnostics.get('method_comparison'))


rho: 813.6584796876001
X:
[[ 1. -0. -0.  0.]
 [-0.  1. -0.  0.]
 [-0. -0.  1.  0.]
 [ 0.  0.  0.  1.]]
Original QAP cost: 720.0
Row sums: [1. 1. 1. 1.]
Column sums: [1. 1. 1. 1.]
Binary residual: 0.0
MDCF status: dca_tolerance


## Finding Lipschits of the gradient

In [26]:


def find_lipschitz_armijo(
    f,
    lower,
    upper,
    n_starts=128,
    max_iter=300,
    alpha0=1.0,
    beta=0.5,
    c=1e-4,
    max_backtracks=25,
    grad_tol=1e-7,
    candidate_tol=1e-4,
    chunk_size=16,
    seed=42,
    device=None,
):
    """
    Multistart projected gradient ascent with Armijo.

    Maximizes phi(x) = ||H_f(x)||_2 over [lower, upper].

    f: callable, R^n -> scalar torch Tensor.

    Returns:
      L_est
      x_best
      candidates
      values
      projected_grad_norms
      device

    Numerical search only; no global guarantee.
    """

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    dtype = torch.float64
    torch.manual_seed(seed)

    lower = torch.as_tensor(lower, dtype=dtype, device=device)
    upper = torch.as_tensor(upper, dtype=dtype, device=device)

    assert lower.ndim == upper.ndim == 1
    assert lower.shape == upper.shape
    assert bool(torch.all(lower < upper))
    assert 0 < beta < 1
    assert 0 < c < 1

    n = lower.numel()
    width = upper - lower

    # ------------------------------------
    # 1. Generate starting points
    # ------------------------------------
    starts = lower + width * torch.rand(
        n_starts, n, device=device, dtype=dtype
    )

    # Random points on boundary faces
    boundary = starts.clone()
    rows = torch.arange(n_starts, device=device)
    coords = torch.randint(n, (n_starts,), device=device)
    sides = torch.rand(n_starts, device=device) > 0.5

    boundary[rows, coords] = torch.where(
        sides, upper[coords], lower[coords]
    )

    starts = torch.cat([starts, boundary], dim=0)

    # All corners for small dimensions
    if n <= 10:
        ids = torch.arange(2**n, device=device)
        bits = (
            (ids[:, None] >> torch.arange(n, device=device)) & 1
        ).bool()
        corners = torch.where(bits, upper, lower)
        starts = torch.cat([starts, corners], dim=0)

    # Center point
    center = ((lower + upper) / 2).unsqueeze(0)
    starts = torch.cat([starts, center], dim=0)

    # ------------------------------------
    # 2. Hessian spectral norm
    # ------------------------------------
    H_single = hessian(f)

    def phi_batch(X):
        H = vmap(H_single)(X)
        eigenvalues = torch.linalg.eigvalsh(H)

        return torch.maximum(
            eigenvalues[:, 0].abs(),
            eigenvalues[:, -1].abs()
        )

    def evaluate(X):
        # Hessian computation requires autodiff
        # even when no derivatives of phi are needed.
        with torch.enable_grad():
            return phi_batch(X).detach()

    def project(X):
        return torch.maximum(
            lower, torch.minimum(upper, X)
        )

    # ------------------------------------
    # 3. Optimize one batch
    # ------------------------------------
    def optimize_batch(initial):
        x = initial.clone().detach()
        B = x.shape[0]

        active = torch.ones(
            B, dtype=torch.bool, device=device
        )

        for _ in range(max_iter):
            if not bool(active.any()):
                break

            # Compute objective and gradient
            with torch.enable_grad():
                z = x.detach().requires_grad_(True)
                values = phi_batch(z)

                grad = torch.autograd.grad(
                    values.sum(), z
                )[0].detach()

            values = values.detach()

            # Projected-gradient stationarity
            pg = project(x + grad) - x
            pg_norm = torch.linalg.vector_norm(pg, dim=1)

            active &= pg_norm > grad_tol

            if not bool(active.any()):
                break

            # --------------------------------
            # Armijo backtracking, per point
            # --------------------------------
            alpha = torch.full(
                (B, 1), alpha0,
                dtype=dtype, device=device
            )

            pending = active.clone()
            accepted = torch.zeros_like(active)
            new_x = x.clone()

            for _ in range(max_backtracks):
                if not bool(pending.any()):
                    break

                trial = project(x + alpha * grad)
                displacement = trial - x

                # Projected Armijo right-hand side
                directional = torch.sum(
                    grad * displacement, dim=1
                )

                # Evaluate only pending candidates
                idx = torch.where(pending)[0]
                trial_values = evaluate(trial[idx])

                rhs = (
                    values[idx]
                    + c * directional[idx]
                )

                step_ok = (
                    (trial_values >= rhs)
                    & (directional[idx] > 0)
                )

                accepted_idx = idx[step_ok]
                new_x[accepted_idx] = trial[accepted_idx]
                accepted[accepted_idx] = True

                # Continue backtracking for failed points
                failed_idx = idx[~step_ok]
                pending[:] = False
                pending[failed_idx] = True
                alpha[failed_idx] *= beta

            # Points that could not make progress stop
            active &= accepted
            x = new_x.detach()

        return x

    # ------------------------------------
    # 4. Optimize all starts in GPU batches
    # ------------------------------------
    endpoints = []

    for chunk in starts.split(chunk_size):
        endpoints.append(optimize_batch(chunk))

    endpoints = torch.cat(endpoints, dim=0)

    # ------------------------------------
    # 5. Evaluate all candidates
    # ------------------------------------
    values = torch.cat([
        evaluate(chunk)
        for chunk in endpoints.split(chunk_size)
    ])

    order = torch.argsort(values, descending=True)

    # Deduplicate by normalized position
    unique_idx = []
    normalized = (endpoints - lower) / width

    for idx in order.tolist():
        if not unique_idx:
            unique_idx.append(idx)
            continue

        distances = torch.linalg.vector_norm(
            normalized[unique_idx] - normalized[idx],
            dim=1
        )

        if bool(torch.all(distances > candidate_tol)):
            unique_idx.append(idx)

    candidates = endpoints[unique_idx]
    candidate_values = values[unique_idx]

    # Recompute stationarity residuals
    residuals = []

    for chunk in candidates.split(chunk_size):
        with torch.enable_grad():
            z = chunk.detach().requires_grad_(True)
            vals = phi_batch(z)
            g = torch.autograd.grad(vals.sum(), z)[0]

        pg = project(z.detach() + g.detach()) - z.detach()
        residuals.append(
            torch.linalg.vector_norm(pg, dim=1)
        )

    residuals = torch.cat(residuals)

    return {
        "L_est": candidate_values[0].item(),
        "x_best": candidates[0].cpu().numpy(),
        "candidates": candidates.cpu().numpy(),
        "values": candidate_values.cpu().numpy(),
        "projected_grad_norms": residuals.cpu().numpy(),
        "device": str(device),
    }



#Finding L

In [27]:

def f(x):
    return x[0]**4 + x[0]*x[1] + 2*x[1]**2

result = find_lipschitz_armijo(
    f,
    lower=[-1, -1],
    upper=[1, 1],
    n_starts=128,
    max_iter=300,
    alpha0=1.0,
    beta=0.5,
    c=1e-4,
    grad_tol=1e-7,
    chunk_size=16,
)

print("Device:", result["device"])
print("Estimated L:", result["L_est"])
print("Best x:", result["x_best"])

# Show the 10 best distinct candidates
for i, (x, L, residual) in enumerate(
    zip(
        result["candidates"][:10],
        result["values"][:10],
        result["projected_grad_norms"][:10],
    ),
    start=1
):
    print(
        f"{i}: x={x}, L={L:.10f}, "
        f"stationarity={residual:.2e}"
    )


/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Device: cpu
Estimated L: 12.123105625617661
Best x: [-1.          0.95088927]
1: x=[-1.          0.95088927], L=12.1231056256, stationarity=0.00e+00
2: x=[ 1.        -0.2629638], L=12.1231056256, stationarity=0.00e+00
3: x=[-1.          0.92329045], L=12.1231056256, stationarity=0.00e+00
4: x=[ 1. -1.], L=12.1231056256, stationarity=0.00e+00
5: x=[-1. -1.], L=12.1231056256, stationarity=0.00e+00
6: x=[1. 1.], L=12.1231056256, stationarity=0.00e+00
7: x=[-1.  1.], L=12.1231056256, stationarity=0.00e+00
8: x=[1.         0.10556001], L=12.1231056256, stationarity=0.00e+00
9: x=[1.         0.02912087], L=12.1231056256, stationarity=0.00e+00
10: x=[-1.          0.28686993], L=12.1231056256, stationarity=0.00e+00



## End-to-end verification: directed 4-city TSP with an **exact comparator**

This small example checks whether the multi-start engine can recover the
best-known cost by checking **all six tours** with the departure city fixed.
It uses the convex quadratic penalty formulation over a reduced Birkhoff polytope,
then prints the winning **initialization method and variant**. The exact comparison
is only a numerical certificate for this 4-city instance; it does **not** certify
MDCF globally on arbitrary inputs.


In [ ]:

import numpy as np
import torch
from itertools import permutations

D_tsp = np.array([
    [0, 4, 7, 10],
    [3, 0, 6, 5],
    [8, 2, 0, 9],
    [4, 11, 1, 0]
], dtype=float)
N_tsp = 4
S_tsp = np.roll(np.eye(N_tsp), 1, axis=1)  # cyclic successor positions
K_tsp = np.kron(S_tsp, D_tsp)
H_tsp = (K_tsp + K_tsp.T) / 2
rho_tsp = np.linalg.eigvalsh(2 * H_tsp)[-1]  # largest eigenvalue of Hessian
B_tsp = torch.tensor(rho_tsp * np.eye(16)-H_tsp,dtype=torch.float64)

def reconstruct_tsp(y):
    Y = y.reshape(2,2)
    last_col = 1-Y.sum(dim=1,keepdim=True)
    top = torch.cat([Y,last_col],dim=1)
    bottom = torch.cat([1-Y.sum(dim=0,keepdim=True),
                        Y.sum().reshape(1,1)-1],dim=1)
    U = torch.cat([top,bottom],dim=0)
    return torch.cat([
        torch.cat([y.new_ones((1,1)),y.new_zeros((1,3))],dim=1),
        torch.cat([y.new_zeros((3,1)),U],dim=1)
    ],dim=0)

A_tsp = []; b_tsp = []
A_tsp.extend(-np.eye(4)); b_tsp.extend([0.]*4)
for i in range(2):
    a=np.zeros((2,2)); a[i,:]=1
    A_tsp.append(a.ravel()); b_tsp.append(1.)
for j in range(2):
    a=np.zeros((2,2)); a[:,j]=1
    A_tsp.append(a.ravel()); b_tsp.append(1.)
A_tsp.append(-np.ones(4)); b_tsp.append(-1.)
A_tsp, b_tsp=np.asarray(A_tsp),np.asarray(b_tsp)

def f_tsp(y):
    P=reconstruct_tsp(y)
    z=P.T.contiguous().reshape(-1)
    return z @ B_tsp @ z

def g_tsp(y):
    return y.new_zeros(())

result_tsp = MDCF.solve_torch(
    f_tsp,g_tsp,[A_tsp],[b_tsp],n=4,
    initialization='all',random_starts=100,random_seed=42,
    return_info=True,max_iter=100,verbose=True
)
P_tsp = reconstruct_tsp(torch.tensor(result_tsp.x,dtype=torch.float64)).detach().numpy()
sequence = list(map(int,np.argmax(P_tsp,axis=0)))
tsp_cost = sum(D_tsp[sequence[k],sequence[(k+1)%4]] for k in range(4))
exact_tsp = min(
    sum(D_tsp[tour[k],tour[k+1]] for k in range(4))
    for perm in permutations(range(1,4))
    for tour in [(0,*perm,0)]
)
print('\n*** TSP VERIFICATION ***')
print('rho (Hessian largest eigenvalue):',rho_tsp)
print('Winning initialization:',result_tsp.diagnostics['winner_method'])
print('Winning variant:',result_tsp.diagnostics['winner_variant'])
print('TSP route:',[i+1 for i in sequence+[sequence[0]]])
print('TSP cost:',tsp_cost)
print('Exact TSP cost:',exact_tsp)
print('Optimality gap (%):',100*(tsp_cost/exact_tsp-1))
print('Feasible permutation:',np.allclose(P_tsp.sum(axis=0),1) and
      np.allclose(P_tsp.sum(axis=1),1) and
      np.max(np.minimum(np.abs(P_tsp),np.abs(1-P_tsp)))<1e-6)
assert tsp_cost >= exact_tsp - 1e-8
